# Explora aquí

Se recomienda utilizar este cuaderno con fines de exploración.

In [14]:
import os
from bs4 import BeautifulSoup
import requests
import time
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [15]:
url = "https://www.compraonline.alcampo.es/categories/bebidas/cervezas/cerveza-lata-est%C3%A1ndar/OC110701"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')
products= soup.find_all('article')

In [16]:
dates_clean = []

for p in products:
    name = p.find('h3')
    price = p.find('span')
    
    if name and price:
        text_price = price.get_text()
        
        price_clean = text_price.replace('$', '').replace('B', '').strip()
        name_clean = name.get_text().strip()
        
        if price_clean != "" and name_clean != "":
            dates_clean.append((name_clean, price_clean))

print(f"Hemos recolectado {len(dates_clean)} productos limpios.")


Hemos recolectado 0 productos limpios.


In [ ]:
conexion = sqlite3.connect("alcampo_cervezas.db")
cursor = conexion.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS cervezas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT,
        precio TEXT
    )
''')

cursor.executemany("INSERT INTO cervezas (nombre, precio) VALUES (?, ?)", dates_clean)

conexion.commit()
conexion.close()

print("¡Datos guardados con éxito en alcampo_cervezas.db!")

In [ ]:
conexion = sqlite3.connect("alcampo_cervezas.db")
df = pd.read_sql_query("SELECT * FROM cervezas", conexion)
conexion.close()

df['precio'] = df['precio'].str.replace(',', '.').astype(float)

In [ ]:
plt.figure(figsize=(10, 6))
top_baratos = df.nsmallest(10, 'precio')
sns.barplot(x='precio', y='nombre', data=top_baratos, palette='viridis')
plt.title('Top 10 Cervezas más Económicas en Alcampo')
plt.xlabel('Precio (€)')
plt.ylabel('Producto')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['precio'], bins=15, kde=True, color='orange')
plt.title('¿Cuánto suelen costar las cervezas? (Distribución)')
plt.xlabel('Rango de Precio (€)')
plt.ylabel('Cantidad de productos')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['precio'], color='skyblue')
plt.title('Dispersión de Precios de Cervezas')
plt.xlabel('Precio (€)')
plt.show()